<a href="https://colab.research.google.com/github/bjanesko/ComputationalChemistryForExperimentalists/blob/main/MolybdenumComplexIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Computational Chemistry for Experimentalists**

Chapter 21: Introduction to Computational Inorganic Chemistry
This worksheet presents a simple example of a computational inorganic chemistry problem.

We compute the structure, orbitals, and infrared spectrum of cis vs trans molybdenum carbonyl complexes

This first part installs and loads the necessary Python packages

In [9]:
!pip install rdkit
!pip install pyscf
!pip install geometric
!pip install pyscf
!pip install pyscf[semiempirical]
!pip install py3Dmol
!pip install xtb

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from pyscf import gto,scf
from pyscf.geomopt.geometric_solver import optimize
import py3Dmol
from xtb.interface import Calculator, Environment
from xtb.libxtb import VERBOSITY_MUTED




  Using cached xtb-22.1.tar.gz (62 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


ModuleNotFoundError: No module named 'xtb'

The next part generates an initial geometry of the  cis and trans complexes.  The initial geometries use ball-and-spring force fields to place the atoms, so the carbonyls show an artifical bent coordination geometry.

In [4]:

mt=Chem.MolFromSmiles('O=[C][Mo]([P](C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3)([C]=O)([C]=O)([C]=O)([P](C4=CC=CC=C4)(C5=CC=CC=C5)C6=CC=CC=C6)')
mc=Chem.MolFromSmiles('[P](C4=CC=CC=C4)(C5=CC=CC=C5)(C6=CC=CC=C6)[Mo@OH1]([P](C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3)([C]=O)([C]=O)([C]=O)([C]=O)')

mt2=Chem.AddHs(mt)
mc2=Chem.AddHs(mc)
AllChem.EmbedMolecule(mc2)
AllChem.MMFFOptimizeMolecule(mc2)
mc3=Chem.MolToMolBlock(mc2)
AllChem.EmbedMolecule(mt2)
AllChem.MMFFOptimizeMolecule(mt2)
mt3=Chem.MolToMolBlock(mt2)
p = py3Dmol.view(width=400,height=200,viewergrid=(1,2))
p.addModel(mc3, 'sdf',viewer=(0,0))
p.addModel(mt3, 'sdf',viewer=(0,1))
p.setStyle({'stick':{},'sphere':{"scale":0.1}})
p.render()

[19:37:05] UFFTYPER: Unrecognized charge state for atom: 0
[19:37:05] UFFTYPER: Unrecognized charge state for atom: 20
[19:37:05] UFFTYPER: Unrecognized charge state for atom: 3
[19:37:05] UFFTYPER: Unrecognized charge state for atom: 28


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

The next part optimizes the geometries of each complex using a simple minimal basis Hartree Fock realization of molecular orbital theory

In [7]:
elements = [atom.GetSymbol() for atom in mt2.GetAtoms()]
coordinates = mt2.GetConformer().GetPositions()
atoms = [(element, coordinate) for element, coordinate in zip(elements, coordinates)]
mt4 = gto.Mole(basis="sto-3g")
mt4.atom = atoms
mt4.build()
#mft=scf.RHF(mt4)
#mft.kernel()
from pyscf.semiempirical import mindo3
mf = mindo3.RMINDO3(mt4)
mol_optimized = optimize(mf)


IndexError: index 42 is out of bounds for axis 0 with size 19